In [1]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd

In [2]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [3]:
# Create df for firm size data
size_df = pd.DataFrame({
    'size': ['Micro (1–9)', 'Small (10–49)', 'Medium (50–249)', 'Large (250+)'],
    'pct': [70, 15, 9, 5]
})

# Add percentage label for plotting
size_df['pct_label'] = size_df['pct'].astype(str) + '%'

# Order size categories small to large for readability
size_order = ['Micro (1–9)', 'Small (10–49)', 'Medium (50–249)', 'Large (250+)']

In [4]:
# Create Chart 1: Firm size composition
# Create bar chart
size_bar = alt.Chart(size_df).mark_bar().encode(
    x=alt.X('size:N',
            sort=size_order,
            title=None,
            axis=alt.Axis(labelAngle=0, grid=False)),
    y=alt.Y('pct:Q',
            title='',
            scale=alt.Scale(domain=[0,100]),
            axis=alt.Axis(labelExpr='datum.value + "%"', 
                          grid=False,
                          values=[0, 20, 40, 60, 80, 100]))
).properties(
    width=500,
    height=250
)
# Add bar labels
labels = size_bar.mark_text(
    align='center',
    baseline='bottom',
    dy=-5,
    fontSize=12
).encode(
    x=alt.X('size:N', sort=size_order),
    y=alt.Y('pct:Q',
            scale=alt.Scale(domain=[0, 100])),
    text=alt.Text('pct_label:N'),
    xOffset=alt.value(40)
)

size_bar = size_bar + labels
size_bar

alt.LayerChart(...)

In [5]:
# Create AI sector study metrics df
growth_raw = pd.DataFrame({
    'year': [2022, 2023, 2024] * 4,
    'metric': (
        ['Employment'] * 3 +
        ['Revenue'] * 3 +
        ['GVA'] * 3 +
        ['Firm Count'] * 3
    ),
    'value': [
        50040, 64539, 86139,       # Employment
        10600, 14200, 23900,       # Revenue (£m)
        3700,  5800,  11800,       # GVA (£m)
        3170,  3713,  5862         # Firm Count
    ]
})

# Index to 2022 = 100
base = growth_raw[growth_raw['year'] == 2022][['metric', 'value']].rename(
    columns={'value': 'base_value'}
)
growth_indexed = growth_raw.merge(base, on='metric')
growth_indexed['indexed'] = (
    growth_indexed['value'] / growth_indexed['base_value'] * 100
).round(1)

# Annotation for GVA 2024
gva_2024 = growth_indexed[
    (growth_indexed['metric'] == 'GVA') &
    (growth_indexed['year'] == 2024)
]

In [6]:
# Create Chart 2: Growth Metrics Indexed (2022 = 100)
growth_line = alt.Chart(growth_indexed).mark_line(
).encode(
    x=alt.X('year:O',
            title=None,
            scale=alt.Scale(padding=0),
            axis=alt.Axis(labelAngle=0, grid=False)),
    y=alt.Y('indexed:Q',
            title='',
            axis=alt.Axis(grid=False)),
    color=alt.Color('metric:N',
                    scale=alt.Scale(
                        domain=['Employment', 'Revenue', 'GVA', 'Firm Count'],),
                    legend=None)
).properties(
    width=500,
    height=250
)

# Add metric labels on GVA 2024 point
labels = growth_indexed[growth_indexed['year'] == 2024]
labels['y_offset'] = labels['metric'].map({
    'Employment': -3,
    'Revenue': 0,
    'GVA': 0,
    'Firm Count': 3
})
labels['indexed_adjusted'] = labels['indexed'] + labels['y_offset']
metric_labels = alt.Chart(labels).mark_text(
    align='left',
    dx=8,
    fontSize=12
).encode(
    x=alt.X('year:O'),
    y=alt.Y('indexed_adjusted:Q'),
    text=alt.Text('metric:N'),
    color=alt.Color('metric:N',
                    scale=alt.Scale(
                        domain=['Employment', 'Revenue', 'GVA', 'Firm Count'],
                    ),
                    legend=None)
)

growth_line = growth_line + metric_labels
growth_line

alt.LayerChart(...)

In [7]:
# Save charts
styles.save(size_bar, 'visualisation', 'ai_firmsize_bar', width=500, height=250)
styles.save(growth_line, 'visualisation', 'ai_growth_line', width=500, height=250)